# Task VII
#### Loading packages


In [ ]:
!pip install scipy numpy pandas matplotlib
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

#### Defining variables

In [ ]:

S0 = float(pd.read_csv("../Data/aapl_28apr.csv")['Close'].iloc[-1])
sigma = 0.3131007174283117 # as calculated before
r = 0.01
T = 0.5
n = 25


#### Calculating in steps
Formula: $R = (1+r\Delta t)$, $u = e^{\sigma \sqrt{\Delta t}}$, $d = e^{-\sigma \sqrt{\Delta t}}$, $q = \frac{R-d}{u-d} $, $\Gamma = qu^2+(1-q)d^2$

In [11]:
R = (1+r*(T/n)) 
u = np.exp(sigma*np.sqrt(T/n))
d = np.exp(-sigma*np.sqrt(T/n))
q = (R-d)/(u-d)
Gamma = q * u**2 + (1-q) * d**2



Formula: $Var^Q(S_n)= S_0^2(\Gamma^n-R^{2n})$, $Var^Q(\bar S_n) = \frac{S_0^2}{(n+1)^2} \sum_{t,\theta = 0}^n\left[ \Gamma^{min(t,\theta)} R^{|t-\theta|} - R^{t+\theta} \right]$, $Cov^Q(S_n,\bar S_n)= \frac{S_0^2}{n+1}\sum_{t=0}^n (\Gamma^tR^{n-t}-R^{n+t})$

In [12]:
Var_Sn = S0**2 * (Gamma**n - R**(2*n))
t_grid = np.arange(0, n+1)
t_index, theta_index = np.meshgrid(t_grid, t_grid, indexing = 'ij')
Var_Snbar = S0**2 / (n+1)**2 * (np.sum(Gamma**np.minimum(t_index, theta_index) * R**(np.abs(t_index - theta_index))- R**(t_index + theta_index)))
Cov = S0**2 / (n+1) * np.sum(Gamma**t_grid * R**(n - t_grid) - R**(n + t_grid))




#### Putting the Mean and the Variance of our payoff together
Formula: $\mu_D=S_0R^n-\frac{S_0}{n+1}\frac{R^{n+1}-1}{R-1}$, $\sigma_D^2 = Var^Q(D_n) = Var^Q(S_n)+Var^Q(\bar S_n) - 2Cov^Q(S_n,\bar S_n)$

In [13]:
mu = S0*R**n - (S0 / (n+1)) * ((R**(n+1)-1) / (R-1))
Var_D = Var_Sn + Var_Snbar - 2*Cov


In [14]:
for name, val in [("S0", S0), 
                  ("u", u), ("d", d), 
                  ("R", R), ("q", q), ("Gamma", Gamma),
                  ("Mü", mu),("Var_Sn", Var_Sn), ("Var_Snbar", Var_Snbar),
                  ("Cov", Cov), ("Var_D", Var_D)]:
    print(f"{name} = {val}")


S0 = 270.7099914550781
u = 1.0452740945330998
d = 0.9566868682866164
R = 1.0002
q = 0.49118968452983786
Gamma = 1.0023613550122803
Mü = 0.6789443987545383
Var_Sn = 3714.628844871287
Var_Snbar = 1195.544232427193
Cov = 1839.8325395059337
Var_D = 1230.5079982866127


#### Approximating the price of the option
Formula: $V_{t=0}^{approx.}=\frac{1}{R^n} \left[\mu_D\Phi\left(\frac{\mu_D}{\sigma_D}\right) + \sigma_D\varphi\left(\frac{\mu_D}{\sigma_D}\right)\right]$

In [15]:
z = mu / np.sqrt(Var_D)

option_price = (1/(R**n)) * (mu * norm.cdf(z) + np.sqrt(Var_D) * norm.pdf(z))
print(option_price)

14.26493290354564
